In [1]:
!pip -q install kagglehub gradio plotly scikit-learn joblib scipy networkx

In [2]:
from pathlib import Path
from collections import defaultdict, deque
import itertools
import io
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import poisson
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content/wc2026_simulator")
PATHS = {
    "data": ROOT / "data",
    "raw": ROOT / "data" / "raw",
    "processed": ROOT / "data" / "processed",
    "models": ROOT / "models",
    "app": ROOT / "app",
    "outputs": ROOT / "outputs",
    "visualizations": ROOT / "visualizations",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print(f"Project ready at: {ROOT}")
print("Folders created:", list(PATHS.keys()))

Project ready at: /content/wc2026_simulator
Folders created: ['data', 'raw', 'processed', 'models', 'app', 'outputs', 'visualizations']


In [3]:
raw_results_path = PATHS["raw"] / "results.csv"
dataset_name = "martj42/international-football-results-from-1872-to-2017"

if raw_results_path.exists():
    results = pd.read_csv(raw_results_path)
    print("Loaded existing results.csv from project folder.")
else:
    try:
        import kagglehub

        downloaded_path = Path(kagglehub.dataset_download(dataset_name))
        candidates = list(downloaded_path.rglob("results.csv"))

        if not candidates:
            raise FileNotFoundError("Could not find results.csv inside downloaded Kaggle dataset.")

        results = pd.read_csv(candidates[0])
        results.to_csv(raw_results_path, index=False)
        print("Downloaded Kaggle dataset successfully.")

    except Exception as e:
        print("Automatic Kaggle download failed.")
        print("Reason:", e)
        print("\nManual fix:")
        print("1. Go to Kaggle.")
        print("2. Download the dataset:", dataset_name)
        print("3. Upload results.csv when prompted below.")

        from google.colab import files
        uploaded = files.upload()

        if "results.csv" not in uploaded:
            raise FileNotFoundError("Please upload a file named results.csv.")

        results = pd.read_csv(io.BytesIO(uploaded["results.csv"]))
        results.to_csv(raw_results_path, index=False)

print("Rows, columns:", results.shape)
display(results.head())

100%|██████████| 1.21M/1.21M [00:01<00:00, 1.09MB/s]

Extracting files...


Downloaded Kaggle dataset successfully.
Rows, columns: (49547, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [4]:
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechia"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

DISPLAY_NAME = {
    "Curacao": "Curaçao",
    "Ivory Coast": "Côte d'Ivoire",
    "United States": "USA",
}

FIFA_RANK = {
    "Spain": 1, "Argentina": 2, "France": 3, "England": 4, "Portugal": 5,
    "Netherlands": 6, "Brazil": 7, "Belgium": 8, "Germany": 10, "Croatia": 11,
    "Morocco": 12, "Colombia": 13, "Mexico": 14, "Uruguay": 15, "United States": 16,
    "Switzerland": 17, "Japan": 18, "Senegal": 19, "Iran": 20, "Sweden": 21,
    "South Korea": 22, "Ecuador": 23, "Turkey": 24, "Australia": 25, "Austria": 26,
    "Canada": 27, "Norway": 29, "Panama": 30, "Scotland": 34, "Egypt": 35,
    "Algeria": 38, "Paraguay": 39, "Ivory Coast": 42, "Czechia": 44, "Tunisia": 46,
    "Uzbekistan": 50, "Qatar": 53, "South Africa": 54, "Iraq": 58, "Saudi Arabia": 59,
    "DR Congo": 60, "Jordan": 66, "Cape Verde": 68, "Ghana": 73,
    "Bosnia and Herzegovina": 74, "Curacao": 82, "Haiti": 84, "New Zealand": 89,
}

CONFEDERATION = {
    "Mexico": "CONCACAF", "South Africa": "CAF", "South Korea": "AFC", "Czechia": "UEFA",
    "Canada": "CONCACAF", "Bosnia and Herzegovina": "UEFA", "Qatar": "AFC", "Switzerland": "UEFA",
    "Brazil": "CONMEBOL", "Morocco": "CAF", "Haiti": "CONCACAF", "Scotland": "UEFA",
    "United States": "CONCACAF", "Paraguay": "CONMEBOL", "Australia": "AFC", "Turkey": "UEFA",
    "Germany": "UEFA", "Curacao": "CONCACAF", "Ivory Coast": "CAF", "Ecuador": "CONMEBOL",
    "Netherlands": "UEFA", "Japan": "AFC", "Sweden": "UEFA", "Tunisia": "CAF",
    "Belgium": "UEFA", "Egypt": "CAF", "Iran": "AFC", "New Zealand": "OFC",
    "Spain": "UEFA", "Cape Verde": "CAF", "Saudi Arabia": "AFC", "Uruguay": "CONMEBOL",
    "France": "UEFA", "Senegal": "CAF", "Iraq": "AFC", "Norway": "UEFA",
    "Argentina": "CONMEBOL", "Algeria": "CAF", "Austria": "UEFA", "Jordan": "AFC",
    "Portugal": "UEFA", "DR Congo": "CAF", "Uzbekistan": "AFC", "Colombia": "CONMEBOL",
    "England": "UEFA", "Croatia": "UEFA", "Ghana": "CAF", "Panama": "CONCACAF",
}

DEFAULT_STYLE = {
    "style": "balanced",
    "attack_style": 1.00,
    "defense_style": 1.00,
    "tempo": 1.00,
    "volatility": 1.00,
}

STYLE_OVERRIDES = {
    "Spain": {"style": "possession control", "attack_style": 1.08, "defense_style": 0.88, "tempo": 0.98, "volatility": 0.82},
    "Argentina": {"style": "balanced elite", "attack_style": 1.08, "defense_style": 0.90, "tempo": 0.96, "volatility": 0.85},
    "France": {"style": "transition power", "attack_style": 1.10, "defense_style": 0.91, "tempo": 1.03, "volatility": 0.88},
    "England": {"style": "structured control", "attack_style": 1.06, "defense_style": 0.91, "tempo": 0.95, "volatility": 0.84},
    "Portugal": {"style": "technical attack", "attack_style": 1.08, "defense_style": 0.92, "tempo": 1.00, "volatility": 0.88},
    "Brazil": {"style": "creative attack", "attack_style": 1.11, "defense_style": 0.96, "tempo": 1.04, "volatility": 0.96},
    "Germany": {"style": "front-foot pressing", "attack_style": 1.08, "defense_style": 0.98, "tempo": 1.04, "volatility": 0.98},
    "Netherlands": {"style": "structured attack", "attack_style": 1.06, "defense_style": 0.92, "tempo": 1.00, "volatility": 0.90},
    "Croatia": {"style": "midfield control", "attack_style": 0.99, "defense_style": 0.92, "tempo": 0.88, "volatility": 0.78},
    "Uruguay": {"style": "compact intensity", "attack_style": 0.99, "defense_style": 0.88, "tempo": 0.93, "volatility": 0.82},
    "Morocco": {"style": "defensive transition", "attack_style": 0.98, "defense_style": 0.87, "tempo": 0.90, "volatility": 0.80},
    "Ecuador": {"style": "defensive low-risk", "attack_style": 0.91, "defense_style": 0.82, "tempo": 0.87, "volatility": 0.76},
    "Switzerland": {"style": "compact balanced", "attack_style": 0.98, "defense_style": 0.91, "tempo": 0.90, "volatility": 0.78},
    "Japan": {"style": "fast technical", "attack_style": 1.03, "defense_style": 0.94, "tempo": 1.06, "volatility": 0.90},
    "Iran": {"style": "deep defensive", "attack_style": 0.93, "defense_style": 0.89, "tempo": 0.86, "volatility": 0.78},
    "Senegal": {"style": "athletic balanced", "attack_style": 1.01, "defense_style": 0.91, "tempo": 0.97, "volatility": 0.86},
    "United States": {"style": "high energy", "attack_style": 1.00, "defense_style": 0.98, "tempo": 1.04, "volatility": 0.94},
    "Mexico": {"style": "possession pressure", "attack_style": 0.99, "defense_style": 0.96, "tempo": 0.97, "volatility": 0.88},
    "Canada": {"style": "direct pace", "attack_style": 1.00, "defense_style": 1.00, "tempo": 1.05, "volatility": 0.98},
}

rows = []
for group, teams in GROUPS.items():
    for team in teams:
        style = DEFAULT_STYLE.copy()
        style.update(STYLE_OVERRIDES.get(team, {}))

        rows.append({
            "team": team,
            "display_name": DISPLAY_NAME.get(team, team),
            "group": group,
            "fifa_rank": FIFA_RANK[team],
            "confederation": CONFEDERATION[team],
            "is_host": team in ["Mexico", "Canada", "United States"],
            **style,
        })

team_meta = pd.DataFrame(rows).sort_values(["group", "fifa_rank"]).reset_index(drop=True)
team_meta.to_csv(PATHS["processed"] / "wc2026_team_metadata.csv", index=False)

print("Teams:", len(team_meta))
display(team_meta)

Teams: 48


,team,display_name,group,fifa_rank,confederation,is_host,style,attack_style,defense_style,tempo,volatility
0,Mexico,Mexico,A,14,CONCACAF,True,possession pressure,0.99,0.96,0.97,0.88
1,South Korea,South Korea,A,22,AFC,False,balanced,1.00,1.00,1.00,1.00
2,Czechia,Czechia,A,44,UEFA,False,balanced,1.00,1.00,1.00,1.00
3,South Africa,South Africa,A,54,CAF,False,balanced,1.00,1.00,1.00,1.00
4,Switzerland,Switzerland,B,17,UEFA,False,compact balanced,0.98,0.91,0.90,0.78
5,Canada,Canada,B,27,CONCACAF,True,direct pace,1.00,1.00,1.05,0.98
6,Qatar,Qatar,B,53,AFC,False,balanced,1.00,1.00,1.00,1.00
7,Bosnia and Herzegovina,Bosnia and Herzegovina,B,74,UEFA,False,balanced,1.00,1.00,1.00,1.00
8,Brazil,Brazil,C,7,CONMEBOL,False,creative attack,1.11,0.96,1.04,0.96
9,Morocco,Morocco,C,12,CAF,False,defensive transition,0.98,0.87,0.90,0.80


In [5]:
ALIASES = {
    "USA": "United States",
    "United States of America": "United States",
    "Korea Republic": "South Korea",
    "Republic of Korea": "South Korea",
    "Czech Republic": "Czechia",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Côte d’Ivoire": "Ivory Coast",
    "Côte d'Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "Cote d’Ivoire": "Ivory Coast",
    "Ivory Coast": "Ivory Coast",
    "Curaçao": "Curacao",
    "Curacao": "Curacao",
    "DR Congo": "DR Congo",
    "Congo DR": "DR Congo",
    "Democratic Republic of Congo": "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",
    "Türkiye": "Turkey",
}

def normalize_team_name(name):
    name = str(name).strip()
    return ALIASES.get(name, name)

results = pd.read_csv(raw_results_path)

required_columns = ["date", "home_team", "away_team", "home_score", "away_score", "tournament", "country", "neutral"]
missing_columns = [c for c in required_columns if c not in results.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

results["date"] = pd.to_datetime(results["date"], errors="coerce")
results["home_team"] = results["home_team"].apply(normalize_team_name)
results["away_team"] = results["away_team"].apply(normalize_team_name)
results["neutral"] = results["neutral"].fillna(False).astype(bool)
results["home_score"] = pd.to_numeric(results["home_score"], errors="coerce")
results["away_score"] = pd.to_numeric(results["away_score"], errors="coerce")

results = results.dropna(subset=["date", "home_team", "away_team", "home_score", "away_score"])
results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)

current_date = pd.Timestamp("2026-05-27")
results = results[(results["date"] >= "2000-01-01") & (results["date"] <= current_date)]
results = results.sort_values("date").reset_index(drop=True)

results.to_csv(PATHS["processed"] / "clean_results.csv", index=False)

all_historical_teams = set(results["home_team"]).union(set(results["away_team"]))
missing_2026_teams = sorted(set(team_meta["team"]) - all_historical_teams)

print("Cleaned rows:", len(results))
print("Date range:", results["date"].min().date(), "to", results["date"].max().date())
print("2026 teams not found in historical dataset:", missing_2026_teams)
display(results.head())

Cleaned rows: 25205
Date range: 2000-01-04 to 2026-05-27
2026 teams not found in historical dataset: []


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False
3,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False
4,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True


In [6]:
GLOBAL_HOME_GOALS = results["home_score"].mean()
GLOBAL_AWAY_GOALS = results["away_score"].mean()
GLOBAL_TEAM_GOALS = pd.concat([results["home_score"], results["away_score"]]).mean()
GLOBAL_POINTS = 1.33

MAJOR_TOURNAMENT_KEYWORDS = [
    "FIFA World Cup",
    "UEFA Euro",
    "Copa América",
    "Copa America",
    "African Cup of Nations",
    "AFC Asian Cup",
    "CONCACAF Gold Cup",
]

def is_major_tournament(tournament):
    tournament = str(tournament)
    return int(any(keyword in tournament for keyword in MAJOR_TOURNAMENT_KEYWORDS))

def is_friendly(tournament):
    return int(str(tournament).lower() == "friendly")

def new_team_state():
    return {
        "elo": 1500.0,
        "gf": deque(maxlen=20),
        "ga": deque(maxlen=20),
        "pts": deque(maxlen=20),
        "played": 0,
    }

team_state = defaultdict(new_team_state)

def avg_or_default(values, default):
    return float(np.mean(values)) if len(values) else float(default)

def make_features(team, opponent, is_home, neutral, tournament):
    own = team_state[team]
    opp = team_state[opponent]

    return {
        "own_elo": own["elo"],
        "opp_elo": opp["elo"],
        "elo_diff": own["elo"] - opp["elo"],
        "gf_recent": avg_or_default(own["gf"], GLOBAL_TEAM_GOALS),
        "ga_recent": avg_or_default(own["ga"], GLOBAL_TEAM_GOALS),
        "form_points": avg_or_default(own["pts"], GLOBAL_POINTS),
        "opp_gf_recent": avg_or_default(opp["gf"], GLOBAL_TEAM_GOALS),
        "opp_ga_recent": avg_or_default(opp["ga"], GLOBAL_TEAM_GOALS),
        "opp_form_points": avg_or_default(opp["pts"], GLOBAL_POINTS),
        "matches_played": own["played"],
        "opp_matches_played": opp["played"],
        "is_home": int(is_home and not neutral),
        "is_neutral": int(neutral),
        "is_major": is_major_tournament(tournament),
        "is_friendly": is_friendly(tournament),
    }

def elo_expected(rating_a, rating_b, home_advantage=0):
    return 1 / (1 + 10 ** ((rating_b - (rating_a + home_advantage)) / 400))

def match_points(goals_for, goals_against):
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0

def update_after_match(home, away, home_goals, away_goals, neutral, tournament):
    home_state = team_state[home]
    away_state = team_state[away]

    home_advantage = 0 if neutral else 60
    expected_home = elo_expected(home_state["elo"], away_state["elo"], home_advantage)

    if home_goals > away_goals:
        actual_home = 1.0
    elif home_goals == away_goals:
        actual_home = 0.5
    else:
        actual_home = 0.0

    goal_margin = abs(home_goals - away_goals)
    margin_multiplier = math.log(goal_margin + 1) + 1

    if is_friendly(tournament):
        k = 14
    elif is_major_tournament(tournament):
        k = 34
    else:
        k = 24

    elo_change = k * margin_multiplier * (actual_home - expected_home)

    home_state["elo"] += elo_change
    away_state["elo"] -= elo_change

    home_state["gf"].append(home_goals)
    home_state["ga"].append(away_goals)
    home_state["pts"].append(match_points(home_goals, away_goals))
    home_state["played"] += 1

    away_state["gf"].append(away_goals)
    away_state["ga"].append(home_goals)
    away_state["pts"].append(match_points(away_goals, home_goals))
    away_state["played"] += 1

training_rows = []

for row in results.itertuples(index=False):
    home = row.home_team
    away = row.away_team
    home_goals = int(row.home_score)
    away_goals = int(row.away_score)
    neutral = bool(row.neutral)
    tournament = row.tournament

    home_features = make_features(home, away, is_home=True, neutral=neutral, tournament=tournament)
    home_features.update({"team": home, "opponent": away, "date": row.date, "goals": home_goals})
    training_rows.append(home_features)

    away_features = make_features(away, home, is_home=False, neutral=neutral, tournament=tournament)
    away_features.update({"team": away, "opponent": home, "date": row.date, "goals": away_goals})
    training_rows.append(away_features)

    update_after_match(home, away, home_goals, away_goals, neutral, tournament)

features_df = pd.DataFrame(training_rows)

FEATURE_COLUMNS = [
    "own_elo", "opp_elo", "elo_diff",
    "gf_recent", "ga_recent", "form_points",
    "opp_gf_recent", "opp_ga_recent", "opp_form_points",
    "matches_played", "opp_matches_played",
    "is_home", "is_neutral", "is_major", "is_friendly",
]

split_date = features_df["date"].quantile(0.85)
train_df = features_df[features_df["date"] < split_date]
valid_df = features_df[features_df["date"] >= split_date]

goal_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", PoissonRegressor(alpha=0.002, max_iter=1000)),
])

goal_model.fit(train_df[FEATURE_COLUMNS], train_df["goals"])

valid_pred = goal_model.predict(valid_df[FEATURE_COLUMNS])
valid_pred = np.clip(valid_pred, 0.05, 6)

mae = mean_absolute_error(valid_df["goals"], valid_pred)
poisson_dev = mean_poisson_deviance(valid_df["goals"], valid_pred)

model_bundle = {
    "model": goal_model,
    "features": FEATURE_COLUMNS,
    "split_date": str(split_date),
}

joblib.dump(model_bundle, PATHS["models"] / "poisson_goal_model.joblib")

strength_rows = []
for team in team_meta["team"]:
    state = team_state[team]
    strength_rows.append({
        "team": team,
        "display_name": DISPLAY_NAME.get(team, team),
        "elo": round(state["elo"], 1),
        "recent_goals_for": round(avg_or_default(state["gf"], GLOBAL_TEAM_GOALS), 2),
        "recent_goals_against": round(avg_or_default(state["ga"], GLOBAL_TEAM_GOALS), 2),
        "recent_points": round(avg_or_default(state["pts"], GLOBAL_POINTS), 2),
        "matches_in_data": state["played"],
        "fifa_rank": FIFA_RANK[team],
    })

team_strength = pd.DataFrame(strength_rows).sort_values(["elo"], ascending=False)
team_strength.to_csv(PATHS["processed"] / "current_team_strength.csv", index=False)

print("Validation MAE:", round(mae, 3))
print("Validation Poisson deviance:", round(poisson_dev, 3))
display(team_strength.head(20))

Validation MAE: 0.93
Validation Poisson deviance: 1.191


,team,display_name,elo,recent_goals_for,recent_goals_against,recent_points,matches_in_data,fifa_rank
28,Spain,Spain,2210.2,2.65,0.95,2.40,340,1
36,Argentina,Argentina,2150.1,2.15,0.50,2.25,340,2
32,France,France,2120.3,2.10,1.05,2.10,348,3
44,England,England,2057.2,2.30,0.55,2.30,316,4
9,Morocco,Morocco,2023.5,1.90,0.30,2.60,305,12
8,Brazil,Brazil,2017.9,1.55,0.90,1.65,369,7
41,Colombia,Colombia,2016.5,1.70,1.10,1.45,321,13
17,Ecuador,Ecuador,2013.4,0.80,0.35,1.55,313,23
21,Japan,Japan,2012.1,2.60,0.45,2.30,395,18
20,Netherlands,Netherlands,1986.6,2.55,1.05,1.90,318,6


In [7]:
META = team_meta.set_index("team").to_dict(orient="index")
HOST_COUNTRIES = {"Mexico", "Canada", "United States"}

GROUP_VENUE_COUNTRY = {
    "A": "Mexico",
    "B": "Canada",
    "C": "United States",
    "D": "United States",
    "E": "Canada",
    "F": "United States",
    "G": "Mexico",
    "H": "United States",
    "I": "Canada",
    "J": "Mexico",
    "K": "United States",
    "L": "Canada",
}

def team_label(team):
    return DISPLAY_NAME.get(team, team)

def fifa_rank_multiplier(team, opponent):
    team_rank = META[team]["fifa_rank"]
    opp_rank = META[opponent]["fifa_rank"]
    diff = opp_rank - team_rank
    return float(np.exp(np.clip(diff, -90, 90) / 260))

def style_multiplier(team, opponent):
    own = META[team]
    opp = META[opponent]
    tempo_factor = ((own["tempo"] + opp["tempo"]) / 2) ** 0.65
    return own["attack_style"] * opp["defense_style"] * tempo_factor

def venue_multiplier(team, opponent, venue_country):
    mult = 1.00
    confed = META[team]["confederation"]

    if team == venue_country and team in HOST_COUNTRIES:
        mult *= 1.13

    if confed == "CONCACAF":
        mult *= 1.02

    if venue_country == "Mexico" and confed not in ["CONCACAF", "CONMEBOL"]:
        mult *= 0.98

    if venue_country == "Canada" and confed in ["AFC", "CAF"]:
        mult *= 0.99

    return mult

def predict_expected_goals(team1, team2, venue_country="United States", neutral=True):
    row1 = pd.DataFrame([make_features(team1, team2, is_home=False, neutral=neutral, tournament="FIFA World Cup")])
    row2 = pd.DataFrame([make_features(team2, team1, is_home=False, neutral=neutral, tournament="FIFA World Cup")])

    base1 = float(goal_model.predict(row1[FEATURE_COLUMNS])[0])
    base2 = float(goal_model.predict(row2[FEATURE_COLUMNS])[0])

    lam1 = base1
    lam1 *= fifa_rank_multiplier(team1, team2)
    lam1 *= style_multiplier(team1, team2)
    lam1 *= venue_multiplier(team1, team2, venue_country)

    lam2 = base2
    lam2 *= fifa_rank_multiplier(team2, team1)
    lam2 *= style_multiplier(team2, team1)
    lam2 *= venue_multiplier(team2, team1, venue_country)

    total = lam1 + lam2
    if total > 5.8:
        scale = 5.8 / total
        lam1 *= scale
        lam2 *= scale

    lam1 = float(np.clip(lam1, 0.15, 4.50))
    lam2 = float(np.clip(lam2, 0.15, 4.50))

    return lam1, lam2

def score_matrix(lambda1, lambda2, max_goals=8):
    goals = np.arange(max_goals + 1)
    p1 = poisson.pmf(goals, lambda1)
    p2 = poisson.pmf(goals, lambda2)
    matrix = np.outer(p1, p2)
    matrix = matrix / matrix.sum()
    return goals, matrix

def match_probabilities(team1, team2, venue_country="United States"):
    lam1, lam2 = predict_expected_goals(team1, team2, venue_country)
    goals, matrix = score_matrix(lam1, lam2)

    team1_win = float(np.tril(matrix, -1).sum())
    draw = float(np.trace(matrix))
    team2_win = float(np.triu(matrix, 1).sum())

    best_idx = np.unravel_index(np.argmax(matrix), matrix.shape)
    most_likely_score = (int(best_idx[0]), int(best_idx[1]))

    return {
        "team1": team1,
        "team2": team2,
        "lambda1": lam1,
        "lambda2": lam2,
        "team1_win_prob": team1_win,
        "draw_prob": draw,
        "team2_win_prob": team2_win,
        "most_likely_score": most_likely_score,
        "goals": goals,
        "matrix": matrix,
    }

def sample_score_from_matrix(matrix, rng):
    flat_index = rng.choice(matrix.size, p=matrix.ravel())
    g1, g2 = np.unravel_index(flat_index, matrix.shape)
    return int(g1), int(g2)

def penalty_win_probability(team1, team2):
    elo_diff = team_state[team1]["elo"] - team_state[team2]["elo"]
    rank_diff = META[team2]["fifa_rank"] - META[team1]["fifa_rank"]
    p = 0.50 + (elo_diff / 1800) + (rank_diff / 900)
    return float(np.clip(p, 0.38, 0.62))

def simulate_match(team1, team2, venue_country="United States", knockout=False, rng=None, strategy="sample"):
    if rng is None:
        rng = np.random.default_rng(SEED)

    pred = match_probabilities(team1, team2, venue_country)
    goals, matrix = pred["goals"], pred["matrix"]

    if strategy == "most_likely":
        g1, g2 = pred["most_likely_score"]
    else:
        g1, g2 = sample_score_from_matrix(matrix, rng)

    result_note = "90 minutes"
    winner = None

    if g1 > g2:
        winner = team1
    elif g2 > g1:
        winner = team2

    if knockout and g1 == g2:
        et_lambda1 = pred["lambda1"] * (30 / 90) * 0.72
        et_lambda2 = pred["lambda2"] * (30 / 90) * 0.72

        et1 = int(rng.poisson(et_lambda1))
        et2 = int(rng.poisson(et_lambda2))

        total1 = g1 + et1
        total2 = g2 + et2

        if total1 > total2:
            winner = team1
            result_note = f"After extra time; ET {et1}-{et2}"
        elif total2 > total1:
            winner = team2
            result_note = f"After extra time; ET {et1}-{et2}"
        else:
            p_team1 = penalty_win_probability(team1, team2)
            winner = team1 if rng.random() < p_team1 else team2
            result_note = f"After penalties; ET {et1}-{et2}"

    return {
        "team1": team1,
        "team2": team2,
        "display_team1": team_label(team1),
        "display_team2": team_label(team2),
        "goals1": g1,
        "goals2": g2,
        "winner": winner,
        "winner_display": team_label(winner) if winner else "Draw",
        "note": result_note,
        "lambda1": pred["lambda1"],
        "lambda2": pred["lambda2"],
        "team1_win_prob": pred["team1_win_prob"],
        "draw_prob": pred["draw_prob"],
        "team2_win_prob": pred["team2_win_prob"],
        "most_likely_score": pred["most_likely_score"],
    }

example = match_probabilities("Ecuador", "Germany", venue_country="Canada")
print("Example: Ecuador vs Germany")
print("Expected goals:", round(example["lambda1"], 2), "-", round(example["lambda2"], 2))
print("Most likely score:", example["most_likely_score"])
print("Win/draw/loss:", round(example["team1_win_prob"], 3), round(example["draw_prob"], 3), round(example["team2_win_prob"], 3))

Example: Ecuador vs Germany
Expected goals: 0.95 - 0.82
Most likely score: (0, 0)
Win/draw/loss: 0.369 0.331 0.3


In [8]:
GROUP_FIXTURE_TEMPLATE = [(0, 1), (2, 3), (0, 2), (1, 3), (0, 3), (1, 2)]

def build_group_table(group, match_df):
    teams = GROUPS[group]
    table = {
        team: {
            "group": group,
            "team": team,
            "display_name": team_label(team),
            "played": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "goal_difference": 0,
            "points": 0,
            "h2h_points": 0,
            "h2h_goal_difference": 0,
            "h2h_goals_for": 0,
            "fifa_rank": META[team]["fifa_rank"],
        }
        for team in teams
    }

    for row in match_df.itertuples(index=False):
        t1, t2 = row.team1, row.team2
        g1, g2 = int(row.goals1), int(row.goals2)

        table[t1]["played"] += 1
        table[t2]["played"] += 1

        table[t1]["goals_for"] += g1
        table[t1]["goals_against"] += g2
        table[t2]["goals_for"] += g2
        table[t2]["goals_against"] += g1

        if g1 > g2:
            table[t1]["wins"] += 1
            table[t2]["losses"] += 1
            table[t1]["points"] += 3
        elif g2 > g1:
            table[t2]["wins"] += 1
            table[t1]["losses"] += 1
            table[t2]["points"] += 3
        else:
            table[t1]["draws"] += 1
            table[t2]["draws"] += 1
            table[t1]["points"] += 1
            table[t2]["points"] += 1

    table_df = pd.DataFrame(table.values())
    table_df["goal_difference"] = table_df["goals_for"] - table_df["goals_against"]

    tied_keys = ["points", "goal_difference", "goals_for"]
    for _, tied_df in table_df.groupby(tied_keys):
        if len(tied_df) < 2:
            continue

        tied_teams = set(tied_df["team"])
        h2h = {team: {"pts": 0, "gd": 0, "gf": 0} for team in tied_teams}

        tied_matches = match_df[
            match_df["team1"].isin(tied_teams) &
            match_df["team2"].isin(tied_teams)
        ]

        for row in tied_matches.itertuples(index=False):
            t1, t2 = row.team1, row.team2
            g1, g2 = int(row.goals1), int(row.goals2)

            h2h[t1]["gf"] += g1
            h2h[t1]["gd"] += g1 - g2
            h2h[t2]["gf"] += g2
            h2h[t2]["gd"] += g2 - g1

            if g1 > g2:
                h2h[t1]["pts"] += 3
            elif g2 > g1:
                h2h[t2]["pts"] += 3
            else:
                h2h[t1]["pts"] += 1
                h2h[t2]["pts"] += 1

        for team in tied_teams:
            idx = table_df.index[table_df["team"] == team][0]
            table_df.loc[idx, "h2h_points"] = h2h[team]["pts"]
            table_df.loc[idx, "h2h_goal_difference"] = h2h[team]["gd"]
            table_df.loc[idx, "h2h_goals_for"] = h2h[team]["gf"]

    table_df = table_df.sort_values(
        [
            "points",
            "goal_difference",
            "goals_for",
            "h2h_points",
            "h2h_goal_difference",
            "h2h_goals_for",
            "fifa_rank",
        ],
        ascending=[False, False, False, False, False, False, True],
    ).reset_index(drop=True)

    table_df.insert(0, "position", np.arange(1, len(table_df) + 1))
    return table_df

def simulate_group_stage(seed=SEED, strategy="sample"):
    rng = np.random.default_rng(seed)
    all_matches = []

    for group, teams in GROUPS.items():
        venue_country = GROUP_VENUE_COUNTRY[group]

        for match_number, (i, j) in enumerate(GROUP_FIXTURE_TEMPLATE, start=1):
            team1 = teams[i]
            team2 = teams[j]

            result = simulate_match(
                team1,
                team2,
                venue_country=venue_country,
                knockout=False,
                rng=rng,
                strategy=strategy,
            )

            all_matches.append({
                "stage": "Group Stage",
                "group": group,
                "match_number": match_number,
                "venue_country": venue_country,
                **result,
            })

    group_matches = pd.DataFrame(all_matches)

    group_tables = {}
    for group in GROUPS:
        group_tables[group] = build_group_table(group, group_matches[group_matches["group"] == group])

    third_rows = []
    qualifiers = {}

    for group, table in group_tables.items():
        qualifiers[f"1{group}"] = table.iloc[0]["team"]
        qualifiers[f"2{group}"] = table.iloc[1]["team"]

        third = table.iloc[2].copy()
        third["seed"] = f"3{group}"
        third_rows.append(third)

    best_thirds = pd.DataFrame(third_rows).sort_values(
        ["points", "goal_difference", "goals_for", "fifa_rank"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    best_thirds.insert(0, "third_rank", np.arange(1, len(best_thirds) + 1))
    best_thirds["qualified"] = best_thirds["third_rank"] <= 8

    for row in best_thirds.itertuples(index=False):
        if row.qualified:
            qualifiers[row.seed] = row.team

    return group_matches, group_tables, best_thirds, qualifiers

group_matches, group_tables, best_thirds, qualifiers = simulate_group_stage(seed=42, strategy="sample")

group_matches.to_csv(PATHS["outputs"] / "group_stage_matches.csv", index=False)
best_thirds.to_csv(PATHS["outputs"] / "best_third_place_teams.csv", index=False)

print("Group-stage matches simulated:", len(group_matches))
print("Qualified teams:", len(qualifiers))
display(group_matches.head(12))
display(group_tables["A"])
display(best_thirds)

Group-stage matches simulated: 72
Qualified teams: 32


,stage,group,match_number,venue_country,team1,team2,display_team1,display_team2,goals1,goals2,winner,winner_display,note,lambda1,lambda2,team1_win_prob,draw_prob,team2_win_prob,most_likely_score
0,Group Stage,A,1,Mexico,Mexico,South Africa,Mexico,South Africa,3,0,Mexico,Mexico,90 minutes,1.980905,0.410899,0.750871,0.182544,0.066585,"(1, 0)"
1,Group Stage,A,2,Mexico,South Korea,Czechia,South Korea,Czechia,1,1,None,Draw,90 minutes,1.645163,0.597863,0.628990,0.239199,0.131811,"(1, 0)"
2,Group Stage,A,3,Mexico,Mexico,South Korea,Mexico,South Korea,2,2,None,Draw,90 minutes,1.211336,0.696182,0.484406,0.302639,0.212955,"(1, 0)"
3,Group Stage,A,4,Mexico,South Africa,Czechia,South Africa,Czechia,1,2,Czechia,Czechia,90 minutes,0.971005,0.977689,0.341533,0.313396,0.345071,"(0, 0)"
4,Group Stage,A,5,Mexico,Mexico,Czechia,Mexico,Czechia,0,1,Czechia,Czechia,90 minutes,2.128414,0.444536,0.767919,0.167558,0.064523,"(2, 0)"
5,Group Stage,A,6,Mexico,South Africa,South Korea,South Africa,South Korea,2,4,South Korea,South Korea,90 minutes,0.552624,1.531146,0.132199,0.254266,0.613534,"(0, 1)"
6,Group Stage,B,1,Canada,Canada,Bosnia and Herzegovina,Canada,Bosnia and Herzegovina,4,0,Canada,Canada,90 minutes,2.745125,0.511744,0.838384,0.115092,0.046524,"(2, 0)"
7,Group Stage,B,2,Canada,Qatar,Switzerland,Qatar,Switzerland,1,2,Switzerland,Switzerland,90 minutes,0.557179,1.767182,0.111102,0.220686,0.668211,"(0, 1)"
8,Group Stage,B,3,Canada,Canada,Qatar,Canada,Qatar,0,2,Qatar,Qatar,90 minutes,1.923337,0.631918,0.681441,0.204978,0.113581,"(1, 0)"
9,Group Stage,B,4,Canada,Bosnia and Herzegovina,Switzerland,Bosnia and Herzegovina,Switzerland,0,3,Switzerland,Switzerland,90 minutes,0.451219,2.522249,0.047629,0.128271,0.824100,"(0, 2)"


,position,group,team,display_name,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,h2h_points,h2h_goal_difference,h2h_goals_for,fifa_rank
0,1,A,Czechia,Czechia,3,2,1,0,4,2,2,7,0,0,0,44
1,2,A,South Korea,South Korea,3,1,2,0,7,5,2,5,0,0,0,22
2,3,A,Mexico,Mexico,3,1,1,1,5,3,2,4,0,0,0,14
3,4,A,South Africa,South Africa,3,0,0,3,3,9,-6,0,0,0,0,54


,third_rank,position,group,team,display_name,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,h2h_points,h2h_goal_difference,h2h_goals_for,fifa_rank,seed,qualified
0,1,3,E,Germany,Germany,3,1,1,1,6,2,4,4,0,0,0,10,3E,True
1,2,3,A,Mexico,Mexico,3,1,1,1,5,3,2,4,0,0,0,14,3A,True
2,3,3,B,Qatar,Qatar,3,1,1,1,5,4,1,4,0,0,0,53,3B,True
3,4,3,C,Morocco,Morocco,3,1,1,1,2,2,0,4,0,0,0,12,3C,True
4,5,3,F,Japan,Japan,3,1,1,1,2,3,-1,4,0,0,0,18,3F,True
5,6,3,I,Norway,Norway,3,1,1,1,2,3,-1,4,0,0,0,29,3I,True
6,7,3,L,Panama,Panama,3,1,0,2,2,2,0,3,0,0,0,30,3L,True
7,8,3,K,Portugal,Portugal,3,1,0,2,5,6,-1,3,0,0,0,5,3K,True
8,9,3,G,Belgium,Belgium,3,1,0,2,2,3,-1,3,0,0,0,8,3G,False
9,10,3,H,Saudi Arabia,Saudi Arabia,3,1,0,2,2,5,-3,3,0,0,0,59,3H,False


In [9]:
R32_TEMPLATE = [
    {"match_id": "M73", "slot1": "2A", "slot2": "2B"},
    {"match_id": "M74", "slot1": "1E", "slot2": "3A/B/C/D/F"},
    {"match_id": "M75", "slot1": "1F", "slot2": "2C"},
    {"match_id": "M76", "slot1": "1C", "slot2": "2F"},
    {"match_id": "M77", "slot1": "1I", "slot2": "3C/D/F/G/H"},
    {"match_id": "M78", "slot1": "2E", "slot2": "2I"},
    {"match_id": "M79", "slot1": "1A", "slot2": "3C/E/F/H/I"},
    {"match_id": "M80", "slot1": "1L", "slot2": "3E/H/I/J/K"},
    {"match_id": "M81", "slot1": "1D", "slot2": "3B/E/F/I/J"},
    {"match_id": "M82", "slot1": "1G", "slot2": "3A/E/H/I/J"},
    {"match_id": "M83", "slot1": "2K", "slot2": "2L"},
    {"match_id": "M84", "slot1": "1H", "slot2": "2J"},
    {"match_id": "M85", "slot1": "1B", "slot2": "3E/F/G/I/J"},
    {"match_id": "M86", "slot1": "1J", "slot2": "2H"},
    {"match_id": "M87", "slot1": "1K", "slot2": "3D/E/I/J/L"},
    {"match_id": "M88", "slot1": "2D", "slot2": "2G"},
]

R16_PAIRS = [
    ("M89", "M74", "M77"),
    ("M90", "M73", "M75"),
    ("M91", "M76", "M78"),
    ("M92", "M79", "M80"),
    ("M93", "M83", "M84"),
    ("M94", "M81", "M82"),
    ("M95", "M86", "M88"),
    ("M96", "M85", "M87"),
]

QF_PAIRS = [
    ("M97", "M89", "M90"),
    ("M98", "M93", "M94"),
    ("M99", "M91", "M92"),
    ("M100", "M95", "M96"),
]

SF_PAIRS = [
    ("M101", "M97", "M98"),
    ("M102", "M99", "M100"),
]

FINAL_PAIR = [("M104", "M101", "M102")]

KNOCKOUT_VENUE_COUNTRY = {
    "M73": "United States", "M74": "United States", "M75": "Mexico", "M76": "United States",
    "M77": "United States", "M78": "United States", "M79": "Mexico", "M80": "United States",
    "M81": "United States", "M82": "United States", "M83": "Canada", "M84": "United States",
    "M85": "Canada", "M86": "United States", "M87": "United States", "M88": "United States",
    "M89": "United States", "M90": "United States", "M91": "United States", "M92": "United States",
    "M93": "United States", "M94": "United States", "M95": "United States", "M96": "United States",
    "M97": "United States", "M98": "United States", "M99": "United States", "M100": "United States",
    "M101": "United States", "M102": "United States", "M104": "United States",
}

def third_slot_groups(slot):
    if not slot.startswith("3"):
        return []
    return slot[1:].split("/")

def assign_third_place_slots(qualified_third_groups):
    slots = []
    for item in R32_TEMPLATE:
        for side in ["slot1", "slot2"]:
            slot = item[side]
            if slot.startswith("3"):
                slots.append({
                    "match_id": item["match_id"],
                    "eligible": third_slot_groups(slot),
                })

    qualified_third_groups = list(qualified_third_groups)

    def backtrack(assignments, remaining_groups, remaining_slots):
        if not remaining_slots:
            return assignments

        remaining_slots = sorted(
            remaining_slots,
            key=lambda s: len(set(s["eligible"]).intersection(remaining_groups))
        )

        slot = remaining_slots[0]
        possible = sorted(set(slot["eligible"]).intersection(remaining_groups))

        for group in possible:
            new_assignments = assignments.copy()
            new_assignments[slot["match_id"]] = group

            result = backtrack(
                new_assignments,
                [g for g in remaining_groups if g != group],
                remaining_slots[1:],
            )

            if result is not None:
                return result

        return None

    assignment = backtrack({}, qualified_third_groups, slots)

    if assignment is None:
        assignment = {}
        remaining = set(qualified_third_groups)
        for slot in slots:
            possible = sorted(set(slot["eligible"]).intersection(remaining))
            if possible:
                chosen = possible[0]
                assignment[slot["match_id"]] = chosen
                remaining.remove(chosen)

    return assignment

def resolve_slot(slot, qualifiers, third_assignment, match_id):
    if slot.startswith("1") or slot.startswith("2"):
        return qualifiers[slot]

    if slot.startswith("3"):
        group = third_assignment[match_id]
        return qualifiers[f"3{group}"]

    raise ValueError(f"Unknown slot: {slot}")

def simulate_knockout_round(pairs, previous_winners, round_name, seed, strategy):
    rng = np.random.default_rng(seed)
    rows = []
    winners = {}

    for match_id, left_id, right_id in pairs:
        team1 = previous_winners[left_id]
        team2 = previous_winners[right_id]
        venue_country = KNOCKOUT_VENUE_COUNTRY.get(match_id, "United States")

        result = simulate_match(
            team1,
            team2,
            venue_country=venue_country,
            knockout=True,
            rng=rng,
            strategy=strategy,
        )

        rows.append({
            "round": round_name,
            "match_id": match_id,
            "venue_country": venue_country,
            **result,
        })
        winners[match_id] = result["winner"]

    return rows, winners

def simulate_knockouts(qualifiers, best_thirds, seed=SEED, strategy="sample"):
    rng = np.random.default_rng(seed)
    knockout_rows = []
    winners = {}

    qualified_third_groups = (
        best_thirds[best_thirds["qualified"]]["group"]
        .astype(str)
        .tolist()
    )
    third_assignment = assign_third_place_slots(qualified_third_groups)

    for item in R32_TEMPLATE:
        match_id = item["match_id"]
        team1 = resolve_slot(item["slot1"], qualifiers, third_assignment, match_id)
        team2 = resolve_slot(item["slot2"], qualifiers, third_assignment, match_id)
        venue_country = KNOCKOUT_VENUE_COUNTRY.get(match_id, "United States")

        result = simulate_match(
            team1,
            team2,
            venue_country=venue_country,
            knockout=True,
            rng=rng,
            strategy=strategy,
        )

        knockout_rows.append({
            "round": "Round of 32",
            "match_id": match_id,
            "seed1": item["slot1"],
            "seed2": item["slot2"],
            "venue_country": venue_country,
            **result,
        })
        winners[match_id] = result["winner"]

    r16_rows, r16_winners = simulate_knockout_round(R16_PAIRS, winners, "Round of 16", seed + 100, strategy)
    knockout_rows.extend(r16_rows)
    winners.update(r16_winners)

    qf_rows, qf_winners = simulate_knockout_round(QF_PAIRS, winners, "Quarterfinal", seed + 200, strategy)
    knockout_rows.extend(qf_rows)
    winners.update(qf_winners)

    sf_rows, sf_winners = simulate_knockout_round(SF_PAIRS, winners, "Semifinal", seed + 300, strategy)
    knockout_rows.extend(sf_rows)
    winners.update(sf_winners)

    final_rows, final_winners = simulate_knockout_round(FINAL_PAIR, winners, "Final", seed + 400, strategy)
    knockout_rows.extend(final_rows)
    winners.update(final_winners)

    knockout_df = pd.DataFrame(knockout_rows)
    champion = winners["M104"]

    return knockout_df, champion, third_assignment

knockout_df, champion, third_assignment = simulate_knockouts(
    qualifiers=qualifiers,
    best_thirds=best_thirds,
    seed=42,
    strategy="sample",
)

knockout_df.to_csv(PATHS["outputs"] / "knockout_results.csv", index=False)

print("Third-place slot assignment:", third_assignment)
print("Predicted champion:", team_label(champion))
display(knockout_df)

Third-place slot assignment: {'M77': 'C', 'M80': 'K', 'M82': 'A', 'M74': 'B', 'M85': 'E', 'M87': 'L', 'M79': 'F', 'M81': 'I'}
Predicted champion: France


,round,match_id,seed1,seed2,venue_country,team1,team2,display_team1,display_team2,goals1,goals2,winner,winner_display,note,lambda1,lambda2,team1_win_prob,draw_prob,team2_win_prob,most_likely_score
0,Round of 32,M73,2A,2B,United States,South Korea,Switzerland,South Korea,Switzerland,2,0,South Korea,South Korea,90 minutes,1.035052,0.921166,0.374101,0.311953,0.313946,"(1, 0)"
1,Round of 32,M74,1E,3A/B/C/D/F,United States,Ivory Coast,Qatar,Côte d'Ivoire,Qatar,1,1,Ivory Coast,Côte d'Ivoire,After extra time; ET 1-0,1.378655,0.707651,0.529889,0.278211,0.191900,"(1, 0)"
2,Round of 32,M75,1F,2C,Mexico,Netherlands,Haiti,Netherlands,Haiti,7,0,Netherlands,Netherlands,90 minutes,2.907099,0.488346,0.860253,0.101243,0.038504,"(2, 0)"
3,Round of 32,M76,1C,2F,United States,Brazil,Sweden,Brazil,Sweden,3,1,Brazil,Brazil,90 minutes,2.212006,0.553869,0.754758,0.167175,0.078067,"(2, 0)"
4,Round of 32,M77,1I,3C/D/F/G/H,United States,France,Morocco,France,Morocco,2,1,France,France,90 minutes,1.165807,0.740947,0.458725,0.307412,0.233863,"(1, 0)"
5,Round of 32,M78,2E,2I,United States,Ecuador,Senegal,Ecuador,Senegal,0,0,Senegal,Senegal,After penalties; ET 0-0,0.918094,0.663501,0.397252,0.351337,0.251411,"(0, 0)"
6,Round of 32,M79,1A,3C/E/F/H/I,Mexico,Czechia,Japan,Czechia,Japan,1,0,Czechia,Czechia,90 minutes,0.466436,2.269044,0.060878,0.154048,0.785073,"(0, 2)"
7,Round of 32,M80,1L,3E/H/I/J/K,United States,Croatia,Portugal,Croatia,Portugal,2,0,Croatia,Croatia,90 minutes,0.874195,1.204533,0.267395,0.295820,0.436785,"(0, 1)"
8,Round of 32,M81,1D,3B/E/F/I/J,United States,United States,Norway,USA,Norway,1,1,United States,USA,After penalties; ET 0-0,1.328835,1.084544,0.422376,0.273152,0.304472,"(1, 1)"
9,Round of 32,M82,1G,3A/E/H/I/J,United States,Iran,Mexico,Iran,Mexico,1,2,Mexico,Mexico,90 minutes,0.626544,0.945589,0.233567,0.350058,0.416374,"(0, 0)"


In [10]:
def simulate_full_tournament(seed=SEED, strategy="sample"):
    group_matches, group_tables, best_thirds, qualifiers = simulate_group_stage(seed=seed, strategy=strategy)
    knockout_df, champion, third_assignment = simulate_knockouts(
        qualifiers=qualifiers,
        best_thirds=best_thirds,
        seed=seed,
        strategy=strategy,
    )

    return {
        "group_matches": group_matches,
        "group_tables": group_tables,
        "best_thirds": best_thirds,
        "qualifiers": qualifiers,
        "knockout_df": knockout_df,
        "champion": champion,
        "third_assignment": third_assignment,
    }

tournament = simulate_full_tournament(seed=42, strategy="sample")

tournament["group_matches"].to_csv(PATHS["outputs"] / "full_group_matches.csv", index=False)
tournament["best_thirds"].to_csv(PATHS["outputs"] / "full_best_thirds.csv", index=False)
tournament["knockout_df"].to_csv(PATHS["outputs"] / "full_knockout.csv", index=False)

for group, table in tournament["group_tables"].items():
    table.to_csv(PATHS["outputs"] / f"group_{group}_table.csv", index=False)

summary = {
    "champion": tournament["champion"],
    "champion_display": team_label(tournament["champion"]),
    "third_assignment": tournament["third_assignment"],
}

with open(PATHS["outputs"] / "tournament_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Full tournament simulation complete.")
print("Predicted champion:", team_label(tournament["champion"]))
print("Outputs saved to:", PATHS["outputs"])
display(tournament["best_thirds"])
display(tournament["knockout_df"])

Full tournament simulation complete.
Predicted champion: France
Outputs saved to: /content/wc2026_simulator/outputs


,third_rank,position,group,team,display_name,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,h2h_points,h2h_goal_difference,h2h_goals_for,fifa_rank,seed,qualified
0,1,3,E,Germany,Germany,3,1,1,1,6,2,4,4,0,0,0,10,3E,True
1,2,3,A,Mexico,Mexico,3,1,1,1,5,3,2,4,0,0,0,14,3A,True
2,3,3,B,Qatar,Qatar,3,1,1,1,5,4,1,4,0,0,0,53,3B,True
3,4,3,C,Morocco,Morocco,3,1,1,1,2,2,0,4,0,0,0,12,3C,True
4,5,3,F,Japan,Japan,3,1,1,1,2,3,-1,4,0,0,0,18,3F,True
5,6,3,I,Norway,Norway,3,1,1,1,2,3,-1,4,0,0,0,29,3I,True
6,7,3,L,Panama,Panama,3,1,0,2,2,2,0,3,0,0,0,30,3L,True
7,8,3,K,Portugal,Portugal,3,1,0,2,5,6,-1,3,0,0,0,5,3K,True
8,9,3,G,Belgium,Belgium,3,1,0,2,2,3,-1,3,0,0,0,8,3G,False
9,10,3,H,Saudi Arabia,Saudi Arabia,3,1,0,2,2,5,-3,3,0,0,0,59,3H,False


,round,match_id,seed1,seed2,venue_country,team1,team2,display_team1,display_team2,goals1,goals2,winner,winner_display,note,lambda1,lambda2,team1_win_prob,draw_prob,team2_win_prob,most_likely_score
0,Round of 32,M73,2A,2B,United States,South Korea,Switzerland,South Korea,Switzerland,2,0,South Korea,South Korea,90 minutes,1.035052,0.921166,0.374101,0.311953,0.313946,"(1, 0)"
1,Round of 32,M74,1E,3A/B/C/D/F,United States,Ivory Coast,Qatar,Côte d'Ivoire,Qatar,1,1,Ivory Coast,Côte d'Ivoire,After extra time; ET 1-0,1.378655,0.707651,0.529889,0.278211,0.191900,"(1, 0)"
2,Round of 32,M75,1F,2C,Mexico,Netherlands,Haiti,Netherlands,Haiti,7,0,Netherlands,Netherlands,90 minutes,2.907099,0.488346,0.860253,0.101243,0.038504,"(2, 0)"
3,Round of 32,M76,1C,2F,United States,Brazil,Sweden,Brazil,Sweden,3,1,Brazil,Brazil,90 minutes,2.212006,0.553869,0.754758,0.167175,0.078067,"(2, 0)"
4,Round of 32,M77,1I,3C/D/F/G/H,United States,France,Morocco,France,Morocco,2,1,France,France,90 minutes,1.165807,0.740947,0.458725,0.307412,0.233863,"(1, 0)"
5,Round of 32,M78,2E,2I,United States,Ecuador,Senegal,Ecuador,Senegal,0,0,Senegal,Senegal,After penalties; ET 0-0,0.918094,0.663501,0.397252,0.351337,0.251411,"(0, 0)"
6,Round of 32,M79,1A,3C/E/F/H/I,Mexico,Czechia,Japan,Czechia,Japan,1,0,Czechia,Czechia,90 minutes,0.466436,2.269044,0.060878,0.154048,0.785073,"(0, 2)"
7,Round of 32,M80,1L,3E/H/I/J/K,United States,Croatia,Portugal,Croatia,Portugal,2,0,Croatia,Croatia,90 minutes,0.874195,1.204533,0.267395,0.295820,0.436785,"(0, 1)"
8,Round of 32,M81,1D,3B/E/F/I/J,United States,United States,Norway,USA,Norway,1,1,United States,USA,After penalties; ET 0-0,1.328835,1.084544,0.422376,0.273152,0.304472,"(1, 1)"
9,Round of 32,M82,1G,3A/E/H/I/J,United States,Iran,Mexico,Iran,Mexico,1,2,Mexico,Mexico,90 minutes,0.626544,0.945589,0.233567,0.350058,0.416374,"(0, 0)"


In [11]:
def plot_match_probabilities(team1, team2, venue_country="United States"):
    pred = match_probabilities(team1, team2, venue_country)

    df = pd.DataFrame({
        "Outcome": [team_label(team1), "Draw", team_label(team2)],
        "Probability": [
            pred["team1_win_prob"],
            pred["draw_prob"],
            pred["team2_win_prob"],
        ],
    })

    fig = px.bar(
        df,
        x="Outcome",
        y="Probability",
        text=df["Probability"].map(lambda x: f"{x:.1%}"),
        title=f"Win Probabilities: {team_label(team1)} vs {team_label(team2)}",
    )
    fig.update_layout(yaxis_tickformat=".0%", height=420)
    fig.update_traces(textposition="outside")
    return fig

def plot_score_heatmap(team1, team2, venue_country="United States"):
    pred = match_probabilities(team1, team2, venue_country)
    goals = pred["goals"]
    matrix = pred["matrix"]

    fig = go.Figure(data=go.Heatmap(
        z=matrix,
        x=[f"{team_label(team2)} {g}" for g in goals],
        y=[f"{team_label(team1)} {g}" for g in goals],
        colorscale="Blues",
        hovertemplate="Score %{y} - %{x}<br>Probability %{z:.2%}<extra></extra>",
    ))

    fig.update_layout(
        title=f"Exact Score Probability Heatmap: {team_label(team1)} vs {team_label(team2)}",
        height=520,
    )
    return fig

def plot_team_strengths():
    df = team_strength.copy()
    df["rank_score"] = 100 - df["fifa_rank"]
    df = df.sort_values("elo", ascending=False).head(24)

    fig = px.scatter(
        df,
        x="elo",
        y="recent_points",
        size="rank_score",
        color="recent_goals_against",
        hover_name="display_name",
        title="Team Strength Map: Elo, Form, Defense, FIFA Rank",
        labels={
            "elo": "Current Elo",
            "recent_points": "Recent points per match",
            "recent_goals_against": "Recent goals conceded",
        },
        color_continuous_scale="RdYlGn_r",
    )
    fig.update_layout(height=560)
    return fig

def plot_knockout_bracket(knockout_df):
    round_order = ["Round of 32", "Round of 16", "Quarterfinal", "Semifinal", "Final"]
    x_map = {round_name: i for i, round_name in enumerate(round_order)}

    fig = go.Figure()

    for round_name in round_order:
        round_df = knockout_df[knockout_df["round"] == round_name].reset_index(drop=True)
        count = len(round_df)

        for i, row in round_df.iterrows():
            y = count - i
            label = (
                f"{row.match_id}<br>"
                f"{row.display_team1} {row.goals1} - {row.goals2} {row.display_team2}<br>"
                f"Winner: {row.winner_display}"
            )

            fig.add_trace(go.Scatter(
                x=[x_map[round_name]],
                y=[y],
                mode="markers+text",
                marker=dict(size=18),
                text=[label],
                textposition="middle right",
                hoverinfo="text",
                showlegend=False,
            ))

    fig.update_layout(
        title="Knockout Bracket Results",
        xaxis=dict(
            tickmode="array",
            tickvals=list(x_map.values()),
            ticktext=round_order,
        ),
        yaxis=dict(showticklabels=False),
        height=900,
        margin=dict(l=20, r=260, t=80, b=40),
    )

    return fig

plot_match_probabilities("Ecuador", "Germany", venue_country="Canada").show()
plot_score_heatmap("Ecuador", "Germany", venue_country="Canada").show()
plot_team_strengths().show()
plot_knockout_bracket(tournament["knockout_df"]).show()

In [12]:
def run_monte_carlo(n_simulations=200, start_seed=1000, strategy="sample"):
    champion_rows = []

    for i in range(n_simulations):
        sim = simulate_full_tournament(seed=start_seed + i, strategy=strategy)
        champion_rows.append({
            "simulation": i + 1,
            "champion": sim["champion"],
            "champion_display": team_label(sim["champion"]),
        })

    champions_df = pd.DataFrame(champion_rows)

    odds = (
        champions_df["champion_display"]
        .value_counts(normalize=True)
        .reset_index()
    )
    odds.columns = ["team", "champion_probability"]
    odds["champion_probability"] = odds["champion_probability"].astype(float)
    odds = odds.sort_values("champion_probability", ascending=False).reset_index(drop=True)

    return champions_df, odds

champions_df, champion_odds = run_monte_carlo(n_simulations=200, start_seed=5000)

champions_df.to_csv(PATHS["outputs"] / "monte_carlo_champions.csv", index=False)
champion_odds.to_csv(PATHS["outputs"] / "champion_probabilities.csv", index=False)

fig = px.bar(
    champion_odds.head(20),
    x="team",
    y="champion_probability",
    text=champion_odds.head(20)["champion_probability"].map(lambda x: f"{x:.1%}"),
    title="Monte Carlo Champion Probabilities",
)
fig.update_layout(yaxis_tickformat=".0%", height=520)
fig.update_traces(textposition="outside")
fig.show()

display(champion_odds.head(20))

,team,champion_probability
0,Spain,0.295
1,Argentina,0.170
2,France,0.125
3,Brazil,0.055
4,England,0.050
5,Portugal,0.035
6,Mexico,0.035
7,Colombia,0.035
8,Iran,0.030
9,Japan,0.025


In [13]:
from pathlib import Path
from IPython.display import HTML, display, FileLink
from google.colab import files
import pandas as pd
import json
import html
from datetime import datetime

# If the tournament object does not exist, create it.
try:
    tournament
except NameError:
    tournament = simulate_full_tournament(seed=42, strategy="sample")

report_path = PATHS["outputs"] / "index.html"

def clean_table(df):
    return df.copy().reset_index(drop=True)

def table_html(df):
    return clean_table(df).to_html(
        index=False,
        border=0,
        classes="data-table",
        escape=False
    )

champion_name = team_label(tournament["champion"])

group_matches = tournament["group_matches"].copy()
group_matches_view = pd.DataFrame({
    "Group": group_matches["group"],
    "Match": group_matches["match_number"],
    "Team 1": group_matches["display_team1"],
    "Score": group_matches["goals1"].astype(str) + " - " + group_matches["goals2"].astype(str),
    "Team 2": group_matches["display_team2"],
    "Winner": group_matches["winner_display"],
    "Venue": group_matches["venue_country"],
    "xG Team 1": group_matches["lambda1"].round(2),
    "xG Team 2": group_matches["lambda2"].round(2),
})

best_thirds = tournament["best_thirds"].copy()
best_thirds_view = best_thirds[[
    "third_rank", "group", "display_name", "played", "wins", "draws",
    "losses", "points", "goal_difference", "goals_for", "goals_against",
    "qualified"
]].rename(columns={
    "third_rank": "Rank",
    "group": "Group",
    "display_name": "Team",
    "played": "P",
    "wins": "W",
    "draws": "D",
    "losses": "L",
    "points": "Pts",
    "goal_difference": "GD",
    "goals_for": "GF",
    "goals_against": "GA",
    "qualified": "Qualified",
})

knockouts = tournament["knockout_df"].copy()
knockouts_view = pd.DataFrame({
    "Round": knockouts["round"],
    "Match": knockouts["match_id"],
    "Team 1": knockouts["display_team1"],
    "Score": knockouts["goals1"].astype(str) + " - " + knockouts["goals2"].astype(str),
    "Team 2": knockouts["display_team2"],
    "Winner": knockouts["winner_display"],
    "Decision": knockouts["note"],
})

group_tables_html = {}
group_matches_html = {}

for group in GROUPS.keys():
    table = tournament["group_tables"][group].copy()
    table_view = table[[
        "position", "display_name", "played", "wins", "draws", "losses",
        "points", "goal_difference", "goals_for", "goals_against"
    ]].rename(columns={
        "position": "Pos",
        "display_name": "Team",
        "played": "P",
        "wins": "W",
        "draws": "D",
        "losses": "L",
        "points": "Pts",
        "goal_difference": "GD",
        "goals_for": "GF",
        "goals_against": "GA",
    })

    group_tables_html[group] = table_html(table_view)
    group_matches_html[group] = table_html(group_matches_view[group_matches_view["Group"] == group])

def make_bracket_html(knockout_df):
    round_order = ["Round of 32", "Round of 16", "Quarterfinal", "Semifinal", "Final"]
    parts = ['<div class="bracket">']

    for round_name in round_order:
        round_df = knockout_df[knockout_df["round"] == round_name]
        parts.append(f'<section class="round-column"><h3>{html.escape(round_name)}</h3>')

        for _, row in round_df.iterrows():
            score = f"{row['display_team1']} {row['goals1']} - {row['goals2']} {row['display_team2']}"
            parts.append(f"""
            <article class="match-card">
                <div class="match-id">{html.escape(str(row['match_id']))}</div>
                <div class="scoreline">{html.escape(score)}</div>
                <div class="winner">Winner: {html.escape(str(row['winner_display']))}</div>
                <div class="note">{html.escape(str(row['note']))}</div>
            </article>
            """)

        parts.append("</section>")

    parts.append("</div>")
    return "\n".join(parts)

all_matches_html = table_html(group_matches_view)
best_thirds_html = table_html(best_thirds_view)
knockouts_html = table_html(knockouts_view)
bracket_html = make_bracket_html(knockouts)

generated_time = datetime.now().strftime("%Y-%m-%d %H:%M")

template = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>FIFA World Cup 2026 Simulator</title>
<meta name="viewport" content="width=device-width, initial-scale=1">

<style>
:root {
    --bg: #f6f7fb;
    --panel: #ffffff;
    --ink: #172033;
    --muted: #667085;
    --line: #e4e7ec;
    --blue: #175cd3;
    --green: #027a48;
    --gold: #b54708;
}

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Inter, Arial, sans-serif;
    background: var(--bg);
    color: var(--ink);
}

header {
    background: linear-gradient(135deg, #09203f, #175cd3);
    color: white;
    padding: 34px 22px;
}

.header-inner {
    max-width: 1180px;
    margin: auto;
}

h1 {
    margin: 0 0 8px;
    font-size: clamp(28px, 4vw, 48px);
}

.subtitle {
    margin: 0;
    color: #dbeafe;
    font-size: 17px;
}

.champion {
    margin-top: 22px;
    display: inline-block;
    background: rgba(255,255,255,0.14);
    border: 1px solid rgba(255,255,255,0.25);
    padding: 14px 18px;
    border-radius: 10px;
    font-size: 22px;
    font-weight: 800;
}

main {
    max-width: 1180px;
    margin: 24px auto 60px;
    padding: 0 16px;
}

.tabs {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    margin-bottom: 16px;
}

.tabs button {
    border: 1px solid var(--line);
    background: white;
    color: var(--ink);
    padding: 10px 14px;
    border-radius: 8px;
    cursor: pointer;
    font-weight: 700;
}

.tabs button.active {
    background: var(--blue);
    color: white;
    border-color: var(--blue);
}

.panel {
    display: none;
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: 10px;
    padding: 18px;
    box-shadow: 0 8px 22px rgba(16, 24, 40, 0.06);
}

.panel.active {
    display: block;
}

.toolbar {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 14px;
}

select {
    padding: 10px 12px;
    border-radius: 8px;
    border: 1px solid var(--line);
    font-weight: 700;
}

.table-wrap {
    overflow-x: auto;
}

.data-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 14px;
}

.data-table th {
    background: #f2f4f7;
    color: #344054;
    text-align: left;
    padding: 11px;
    border-bottom: 1px solid var(--line);
    white-space: nowrap;
}

.data-table td {
    padding: 10px 11px;
    border-bottom: 1px solid var(--line);
    white-space: nowrap;
}

.data-table tr:hover td {
    background: #f9fafb;
}

.section-title {
    margin: 4px 0 14px;
}

.bracket {
    display: grid;
    grid-template-columns: repeat(5, minmax(220px, 1fr));
    gap: 14px;
    overflow-x: auto;
}

.round-column h3 {
    margin: 0 0 10px;
}

.match-card {
    background: #f9fafb;
    border: 1px solid var(--line);
    border-left: 4px solid var(--blue);
    border-radius: 8px;
    padding: 10px;
    margin-bottom: 10px;
}

.match-id {
    color: var(--muted);
    font-size: 12px;
    font-weight: 800;
}

.scoreline {
    font-weight: 800;
    margin-top: 5px;
}

.winner {
    color: var(--green);
    margin-top: 5px;
    font-weight: 800;
}

.note {
    color: var(--muted);
    margin-top: 4px;
    font-size: 12px;
}

footer {
    color: var(--muted);
    text-align: center;
    padding: 30px 16px;
}

@media (max-width: 800px) {
    .bracket {
        grid-template-columns: 1fr;
    }
}
</style>
</head>

<body>
<header>
    <div class="header-inner">
        <h1>FIFA World Cup 2026 Simulator</h1>
        <p class="subtitle">Predicted group stage, tables, best third-place teams, knockouts, and champion.</p>
        <div class="champion">Predicted Champion: __CHAMPION__</div>
    </div>
</header>

<main>
    <nav class="tabs">
        <button class="tab-button active" onclick="showTab('matches')">All Matches</button>
        <button class="tab-button" onclick="showTab('groups')">Group Tables</button>
        <button class="tab-button" onclick="showTab('thirds')">Best Thirds</button>
        <button class="tab-button" onclick="showTab('knockouts')">Knockouts</button>
        <button class="tab-button" onclick="showTab('bracket')">Bracket</button>
    </nav>

    <section id="matches" class="panel active">
        <h2 class="section-title">All Group Matches</h2>
        <div class="table-wrap">__ALL_MATCHES__</div>
    </section>

    <section id="groups" class="panel">
        <h2 class="section-title">Group Tables</h2>
        <div class="toolbar">
            <label for="groupSelect"><strong>Choose group:</strong></label>
            <select id="groupSelect" onchange="updateGroup()">
                <option>A</option><option>B</option><option>C</option><option>D</option>
                <option>E</option><option>F</option><option>G</option><option>H</option>
                <option>I</option><option>J</option><option>K</option><option>L</option>
            </select>
        </div>

        <h3>Table</h3>
        <div id="groupTable" class="table-wrap"></div>

        <h3>Matches</h3>
        <div id="groupMatches" class="table-wrap"></div>
    </section>

    <section id="thirds" class="panel">
        <h2 class="section-title">Best Third-Place Teams</h2>
        <p>The top 8 third-place teams qualify for the Round of 32.</p>
        <div class="table-wrap">__BEST_THIRDS__</div>
    </section>

    <section id="knockouts" class="panel">
        <h2 class="section-title">Knockout Results</h2>
        <div class="table-wrap">__KNOCKOUTS__</div>
    </section>

    <section id="bracket" class="panel">
        <h2 class="section-title">Knockout Bracket</h2>
        __BRACKET__
    </section>
</main>

<footer>
    Generated on __GENERATED_TIME__. This is a probabilistic simulation, not a guaranteed forecast.
</footer>

<script>
const groupTables = __GROUP_TABLES_JSON__;
const groupMatches = __GROUP_MATCHES_JSON__;

function showTab(id) {
    document.querySelectorAll('.panel').forEach(panel => {
        panel.classList.remove('active');
    });

    document.querySelectorAll('.tab-button').forEach(button => {
        button.classList.remove('active');
    });

    document.getElementById(id).classList.add('active');
    event.target.classList.add('active');
}

function updateGroup() {
    const group = document.getElementById('groupSelect').value;
    document.getElementById('groupTable').innerHTML = groupTables[group];
    document.getElementById('groupMatches').innerHTML = groupMatches[group];
}

updateGroup();
</script>
</body>
</html>
"""

html_report = template
html_report = html_report.replace("__CHAMPION__", html.escape(champion_name))
html_report = html_report.replace("__ALL_MATCHES__", all_matches_html)
html_report = html_report.replace("__BEST_THIRDS__", best_thirds_html)
html_report = html_report.replace("__KNOCKOUTS__", knockouts_html)
html_report = html_report.replace("__BRACKET__", bracket_html)
html_report = html_report.replace("__GENERATED_TIME__", generated_time)
html_report = html_report.replace("__GROUP_TABLES_JSON__", json.dumps(group_tables_html))
html_report = html_report.replace("__GROUP_MATCHES_JSON__", json.dumps(group_matches_html))

report_path.write_text(html_report, encoding="utf-8")

print("Static interactive dashboard created:")
print(report_path)

display(FileLink(str(report_path)))

# This downloads the file so you can upload it to Netlify Drop, GitHub Pages, or Google Drive.
files.download(str(report_path))

Static interactive dashboard created:
/content/wc2026_simulator/outputs/index.html


/content/wc2026_simulator/outputs/index.html

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>